# Depth-Weighted Perturbation Theory: Exploring $\mathcal{N}_\lambda$ for QED $g-2$

This notebook tests the primary target identified in [Motivation_of_Expansions.md](Motivation_of_Expansions.md):
applying the graded SCN attenuation operator $\mathcal{N}_\lambda(x) = \lambda^{-\sigma(x)} x$
to the QED perturbative expansion for the electron anomalous magnetic moment $a_e = (g-2)/2$.

**Two approaches:**
1. **Loop-order attenuation:** $\sigma = $ loop order $n$. Each loop contributes with weight $\lambda^{-n}$.
2. **Nesting-depth attenuation:** $\sigma = $ number of self-energy sub-insertions. Skeleton diagrams keep full weight; nested SE diagrams are suppressed.

**Success criterion:** A single $\lambda$ that fits experiment consistently across loop orders.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.graded_scn import (
    QED_G2_COEFFICIENTS, ALPHA, ALPHA_OVER_PI,
    A_E_EXPERIMENT, A_E_UNCERTAINTY,
    compute_g2_standard, compute_g2_attenuated, compute_g2_threshold,
    fit_lambda_to_experiment, consistency_test, attenuated_series_analysis,
)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## 1. The QED Perturbative Series

The electron anomalous magnetic moment is computed as:
$$a_e = \sum_{n=1}^{\infty} C_n \left(\frac{\alpha}{\pi}\right)^n$$

where the coefficients $C_n$ are known through 5-loop.

In [ ]:
print(f"Fine structure constant: α = 1/{1/ALPHA:.6f}")
print(f"α/π = {ALPHA_OVER_PI:.6e}")
print()
print(f"{'Loop':>4}  {'C_n':>18}  {'C_n × (α/π)^n':>18}  {'Cumulative a_e':>18}  {'Source'}")
print("-" * 100)

cumulative = 0.0
for n in range(1, 6):
    c = QED_G2_COEFFICIENTS[n]
    term = c['value'] * ALPHA_OVER_PI**n
    cumulative += term
    print(f"{n:>4}  {c['value']:>+18.12f}  {term:>+18.6e}  {cumulative:>18.15e}  {c['source']}")

print()
print(f"Standard QED (5-loop): {cumulative:.15e}")
print(f"Experiment:            {A_E_EXPERIMENT:.15e}")
print(f"Difference:            {cumulative - A_E_EXPERIMENT:+.4e}")
print(f"Significance:          {(cumulative - A_E_EXPERIMENT)/A_E_UNCERTAINTY:+.1f}σ")
print()
print("Note: The ~37σ gap reflects our truncated numerical series vs. the published")
print("total (which includes higher-precision α and sub-leading corrections).")
print("The published QED total is 0.00115965218178(77), within 1σ of experiment.")

## 2. Approach 1: Loop-Order Attenuation

Define $\sigma(\text{diagram}) = n$ (loop order). The attenuated series is:
$$a_e(\lambda) = \sum_{n=1}^{N} C_n \left(\frac{\alpha}{\pi}\right)^n \lambda^{-n} = \sum_{n=1}^{N} C_n \left(\frac{\alpha}{\pi\lambda}\right)^n$$

This is equivalent to evaluating the standard series at an effective coupling $\alpha_{\text{eff}} = \alpha/\lambda$.

- $\lambda = 1$: standard QED
- $\lambda \to \infty$: tree-level ($a_e = 0$)
- $\lambda < 1$: enhanced coupling (non-physical for the SCN interpretation)

In [ ]:
# Compute a_e(λ) over a range of λ
lambdas = np.linspace(0.95, 1.10, 2000)
a_e_values = np.array([compute_g2_attenuated(l, 5).a_e_weighted for l in lambdas])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: a_e(λ) vs λ
ax1.plot(lambdas, a_e_values * 1e3, 'b-', linewidth=2, label=r'$a_e(\lambda)$')
ax1.axhline(A_E_EXPERIMENT * 1e3, color='r', linestyle='--', linewidth=1.5, label='Experiment')
ax1.axhline(compute_g2_standard(5) * 1e3, color='g', linestyle=':', linewidth=1.5, label='Standard QED (5-loop)')
ax1.set_xlabel(r'$\lambda$')
ax1.set_ylabel(r'$a_e \times 10^3$')
ax1.set_title(r'$a_e(\lambda)$ — Loop-Order Attenuation')
ax1.legend()

# Right: |Δ| vs λ (log scale)
residuals = np.abs(a_e_values - A_E_EXPERIMENT)
ax2.semilogy(lambdas, residuals, 'b-', linewidth=2)
ax2.axvline(1.0, color='k', linestyle=':', alpha=0.5, label=r'$\lambda = 1$ (standard)')
best_idx = np.argmin(residuals)
ax2.plot(lambdas[best_idx], residuals[best_idx], 'ro', markersize=10, 
         label=f'Minimum: λ={lambdas[best_idx]:.4f}')
ax2.set_xlabel(r'$\lambda$')
ax2.set_ylabel(r'$|a_e(\lambda) - a_e^{\mathrm{exp}}|$')
ax2.set_title('Residual vs. Experiment')
ax2.legend()

plt.tight_layout()
plt.savefig('../plots/g2_loop_order_attenuation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOptimal λ = {lambdas[best_idx]:.6f}")
print(f"This is {abs(1 - lambdas[best_idx])*100:.3f}% away from λ=1 (standard QED).")
print(f"Minimum residual = {residuals[best_idx]:.4e} = {residuals[best_idx]/A_E_UNCERTAINTY:.0f}σ")

In [ ]:
# Wider view: a_e(λ) over λ ∈ [0.5, 5]
lambdas_wide = np.linspace(0.5, 5.0, 5000)
a_e_wide = np.array([compute_g2_attenuated(l, 5).a_e_weighted for l in lambdas_wide])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lambdas_wide, a_e_wide * 1e3, 'b-', linewidth=2)
ax.axhline(A_E_EXPERIMENT * 1e3, color='r', linestyle='--', linewidth=1.5, label='Experiment')
ax.axvline(1.0, color='k', linestyle=':', alpha=0.5, label=r'$\lambda=1$')
ax.set_xlabel(r'$\lambda$', fontsize=14)
ax.set_ylabel(r'$a_e \times 10^3$', fontsize=14)
ax.set_title(r'$a_e(\lambda)$ over full range — monotonically decreasing', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("The function is essentially monotone decreasing (dominated by C_1(α/πλ)),")
print("so there is at most one crossing of the experimental value, at λ ≈ 1.")
print("Loop-order attenuation is just a rescaling of α — trivially optimal at λ=1.")

### 2.1 Why loop-order attenuation is trivial

The series $a_e(\lambda) = \sum C_n (\alpha/\pi\lambda)^n$ is just the standard series evaluated at $\alpha_{\text{eff}} = \alpha/\lambda$. Since QED perturbation theory converges beautifully for $a_e$ (the 1-loop Schwinger term provides 99.8% of the total), any $\lambda \neq 1$ simply shifts the effective coupling away from its physical value.

**Verdict:** Loop-order attenuation cannot improve on standard QED. The optimal $\lambda$ is trivially 1.

## 3. Approach 2: Nesting-Depth Attenuation

A more interesting application: define $\sigma$ as the **self-referential nesting depth** within each diagram, not the loop order.

At 2-loop, the 7 vertex diagrams decompose into:
- **Skeleton** (σ_nest = 0): II, III(a), III(b), III(c) — no SE sub-insertions.
  Combined contribution: $C_2^{\text{skel}}$
- **SE-insertion** (σ_nest = 1): I(a), I(b), I(c) — contain one electron self-energy insertion.
  Combined contribution: $C_2^{\text{SE}}$

Under nesting-depth attenuation:
$$C_2(\lambda) = C_2^{\text{skel}} + C_2^{\text{SE}} \cdot \lambda^{-1}$$

- $\lambda = 1$: standard QED ($C_2 = C_2^{\text{skel}} + C_2^{\text{SE}}$)
- $\lambda \to \infty$: binary SCN ($C_2 = C_2^{\text{skel}}$ — SE diagrams killed)

This is **not** a trivial rescaling of $\alpha$ — it changes the *internal structure* of each loop order.

In [ ]:
# 2-loop coefficient decomposition
# From topology.py and amplitudes.py:
#   C_2 = -0.328478965579193  (total)
#   SE contribution (I(a) + I(b) + I(c)) = +0.7714
#   Skeleton contribution = C_2 - C_2^SE

C2_TOTAL = QED_G2_COEFFICIENTS[2]['value']
C2_SE = 0.7714  # SE-insertion diagrams I(a), I(b), I(c)
C2_SKEL = C2_TOTAL - C2_SE  # Skeleton diagrams II, III(a-c)

print(f"2-loop coefficient decomposition:")
print(f"  C_2 (total)    = {C2_TOTAL:+.12f}")
print(f"  C_2 (SE)       = {C2_SE:+.12f}   [I(a), I(b), I(c) — nesting depth 1]")
print(f"  C_2 (skeleton) = {C2_SKEL:+.12f}   [II, III(a), III(b), III(c) — nesting depth 0]")
print(f"  Check: {C2_SKEL:+.6f} + {C2_SE:+.6f} = {C2_SKEL + C2_SE:+.12f}")
print()
print(f"Note: SE diagrams contribute +0.77, skeleton diagrams contribute -1.10.")
print(f"The cancellation between them produces the small C_2 ≈ -0.33.")
print(f"This cancellation is what binary SCN destroys.")

In [ ]:
def compute_g2_nesting_attenuated(lambda_val, max_loop=5):
    """
    Compute a_e with nesting-depth attenuation.
    
    At 2-loop: SE-insertion diagrams get weight λ^{-1}, skeleton get weight 1.
    At other loops: full coefficient used (nesting decomposition unavailable).
    """
    total = 0.0
    terms = {}
    
    for n in range(1, max_loop + 1):
        if n not in QED_G2_COEFFICIENTS:
            continue
        
        if n == 2:
            # Decompose into skeleton + SE with different weights
            c_eff = C2_SKEL + C2_SE * lambda_val**(-1)
        else:
            # No decomposition available — use full coefficient
            c_eff = QED_G2_COEFFICIENTS[n]['value']
        
        term = c_eff * ALPHA_OVER_PI**n
        terms[n] = term
        total += term
    
    return total, terms


# Compute over a range of λ
lambdas_nest = np.linspace(0.5, 20.0, 5000)
a_e_nest = np.array([compute_g2_nesting_attenuated(l)[0] for l in lambdas_nest])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: a_e(λ) vs λ
ax1.plot(lambdas_nest, a_e_nest * 1e3, 'b-', linewidth=2, label=r'$a_e(\lambda)$ — nesting attenuation')
ax1.axhline(A_E_EXPERIMENT * 1e3, color='r', linestyle='--', linewidth=1.5, label='Experiment')
standard_5l = compute_g2_standard(5)
ax1.axhline(standard_5l * 1e3, color='g', linestyle=':', linewidth=1.5, label='Standard QED')
ax1.axvline(1.0, color='k', linestyle=':', alpha=0.5)
ax1.set_xlabel(r'$\lambda$', fontsize=14)
ax1.set_ylabel(r'$a_e \times 10^3$', fontsize=14)
ax1.set_title('Nesting-Depth Attenuation')
ax1.legend()

# Right: residual
resid_nest = np.abs(a_e_nest - A_E_EXPERIMENT)
ax2.semilogy(lambdas_nest, resid_nest, 'b-', linewidth=2)
ax2.axvline(1.0, color='k', linestyle=':', alpha=0.5, label=r'$\lambda=1$')
best_nest_idx = np.argmin(resid_nest)
ax2.plot(lambdas_nest[best_nest_idx], resid_nest[best_nest_idx], 'ro', markersize=10,
         label=f'Minimum: λ={lambdas_nest[best_nest_idx]:.3f}')
ax2.set_xlabel(r'$\lambda$', fontsize=14)
ax2.set_ylabel(r'$|a_e(\lambda) - a_e^{\mathrm{exp}}|$')
ax2.set_title('Residual from Experiment')
ax2.legend()

plt.tight_layout()
plt.savefig('../plots/g2_nesting_attenuation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOptimal λ (nesting) = {lambdas_nest[best_nest_idx]:.4f}")
print(f"a_e at optimal: {a_e_nest[best_nest_idx]:.15e}")
print(f"Residual: {resid_nest[best_nest_idx]:.4e} = {resid_nest[best_nest_idx]/A_E_UNCERTAINTY:.1f}σ")

In [ ]:
# Zoom into the structure: how does C_2(λ) change?
lambdas_c2 = np.linspace(0.5, 10.0, 1000)
c2_eff = C2_SKEL + C2_SE / lambdas_c2

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(lambdas_c2, c2_eff, 'b-', linewidth=2)
ax.axhline(C2_TOTAL, color='g', linestyle=':', linewidth=1.5, label=f'Standard C₂ = {C2_TOTAL:.4f}')
ax.axhline(C2_SKEL, color='r', linestyle='--', linewidth=1.5, label=f'Binary SCN C₂ = {C2_SKEL:.4f} (λ→∞)')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(1.0, color='k', linestyle=':', alpha=0.5)
ax.set_xlabel(r'$\lambda$', fontsize=14)
ax.set_ylabel(r'$C_2(\lambda)$', fontsize=14)
ax.set_title(r'Effective 2-loop coefficient under nesting-depth attenuation', fontsize=14)
ax.legend(fontsize=12)
ax.annotate(f'λ=1: C₂ = {C2_TOTAL:.4f}', xy=(1.0, C2_TOTAL), xytext=(2.5, C2_TOTAL + 0.15),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=11, color='green')
plt.tight_layout()
plt.savefig('../plots/g2_c2_effective.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"As λ increases from 1 to ∞:")
print(f"  C_2 changes from {C2_TOTAL:.6f} to {C2_SKEL:.6f}")
print(f"  The SE contribution ({C2_SE:+.4f}) is progressively suppressed.")
print(f"  At λ=2: C_2 = {C2_SKEL + C2_SE/2:.6f}")
print(f"  At λ=5: C_2 = {C2_SKEL + C2_SE/5:.6f}")

## 4. The Scale of 2-Loop Effects

Before interpreting the $\lambda$ fits, let's understand *how much* the 2-loop term matters relative to the experimental precision.

In [ ]:
# Size of each loop contribution vs experimental uncertainty
print(f"{'Loop':>4}  {'Term size':>15}  {'In units of σ_exp':>18}")
print("-" * 50)
for n in range(1, 6):
    c = QED_G2_COEFFICIENTS[n]['value']
    term = abs(c * ALPHA_OVER_PI**n)
    print(f"{n:>4}  {term:>15.6e}  {term/A_E_UNCERTAINTY:>18.0f}σ")

print()
print(f"Experimental uncertainty: σ_exp = {A_E_UNCERTAINTY:.2e}")
print()

# How much does the nesting decomposition shift things at 2-loop?
c2_std_term = C2_TOTAL * ALPHA_OVER_PI**2
c2_skel_term = C2_SKEL * ALPHA_OVER_PI**2
c2_se_term = C2_SE * ALPHA_OVER_PI**2
print(f"2-loop total contribution:   {c2_std_term:+.6e}")
print(f"  Skeleton part:             {c2_skel_term:+.6e}  ({c2_skel_term/A_E_UNCERTAINTY:+.0f}σ)")
print(f"  SE-insertion part:         {c2_se_term:+.6e}  ({c2_se_term/A_E_UNCERTAINTY:+.0f}σ)")
print()
print(f"Removing the SE part (binary SCN) shifts a_e by {-c2_se_term:+.6e} = {-c2_se_term/A_E_UNCERTAINTY:+.0f}σ")
print(f"This is the 415σ falsification from the binary analysis.")

## 5. Consistency Across Truncation Orders

For loop-order attenuation, test whether $\lambda^*$ stabilizes as we include more loops.

In [ ]:
# High-resolution fit near λ=1 for each truncation order
print(f"{'max_loop':>8}  {'λ*':>10}  {'a_e(λ*)':>20}  {'Residual (σ)':>14}  {'Δ from λ=1':>12}")
print("-" * 75)

for ml in [2, 3, 4, 5]:
    # Fine search near λ=1
    fit = fit_lambda_to_experiment(max_loop=ml, lambda_range=(0.999, 1.001), n_points=50000)
    lam = fit['lambda_optimal']
    ae = fit['a_e_at_optimal']
    sig = fit['residual_sigma']
    print(f"{ml:>8}  {lam:>10.7f}  {ae:>20.15e}  {sig:>+14.1f}  {lam - 1:>+12.7f}")

print()
print("λ* ≈ 1 at all truncation orders — the standard series is already optimal.")
print("The small deviations from λ=1 reflect sub-leading effects (hadronic, EW corrections)")
print("that our pure-QED coefficients don't capture.")

## 6. Per-Loop Contribution Analysis

Visualize how attenuation changes the relative importance of each loop order.

In [ ]:
# Compare standard vs attenuated at a few λ values
lambda_test_values = [1.0, 1.5, 2.0, 5.0]

fig, axes = plt.subplots(1, len(lambda_test_values), figsize=(16, 5), sharey=True)

for ax, lam in zip(axes, lambda_test_values):
    analysis = attenuated_series_analysis(lam, max_loop=5)
    loops = list(analysis['per_loop'].keys())
    std_terms = [analysis['per_loop'][n]['standard_term'] for n in loops]
    att_terms = [analysis['per_loop'][n]['attenuated_term'] for n in loops]
    
    x = np.arange(len(loops))
    width = 0.35
    ax.bar(x - width/2, [abs(t) for t in std_terms], width, label='Standard', alpha=0.7)
    ax.bar(x + width/2, [abs(t) for t in att_terms], width, label='Attenuated', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{n}L' for n in loops])
    ax.set_yscale('log')
    ax.set_title(f'λ = {lam}')
    if ax == axes[0]:
        ax.set_ylabel('|contribution to $a_e$|')
        ax.legend(fontsize=9)

fig.suptitle('Per-loop contributions: standard vs. attenuated', fontsize=14)
plt.tight_layout()
plt.savefig('../plots/g2_per_loop_contributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Phase Diagram: Threshold $k$ vs Attenuation $\lambda$

The graded SCN framework provides two controls:
- **Hard threshold** $k$: retain only loops $< k$
- **Soft attenuation** $\lambda$: weight loop $n$ by $\lambda^{-n}$

Explore the 2D parameter space.

In [ ]:
# Threshold operator: a_e at each cutoff k
print(f"{'k':>3}  {'Loops kept':>12}  {'a_e':>20}  {'Δ from exp (σ)':>16}  {'Note'}")
print("-" * 80)

for k in range(1, 7):
    r = compute_g2_threshold(k)
    note = ""
    if k == 1:
        note = "tree-level (binary SCN at loop level)"
    elif k == 2:
        note = "Schwinger only"
    elif k == 6:
        note = "all 5 loops (standard)"
    print(f"{k:>3}  {min(k-1, 5):>12}  {r.a_e_weighted:>20.15e}  {r.delta_sigma:>+16.1f}  {note}")

print()
print("Every threshold k ≥ 2 gives essentially the same answer (dominated by 1-loop).")
print("The corrections from 2-loop onward are at the 10⁻⁶ level of a_e.")

In [ ]:
# 2D scan: threshold k × attenuation λ
k_values = [2, 3, 4, 5, 6]
lambda_values = np.linspace(0.9, 2.0, 200)

fig, ax = plt.subplots(figsize=(10, 6))

for k in k_values:
    a_e_kl = []
    for lam in lambda_values:
        total = 0.0
        for n in range(1, min(k, 6)):
            if n in QED_G2_COEFFICIENTS:
                total += QED_G2_COEFFICIENTS[n]['value'] * ALPHA_OVER_PI**n * lam**(-n)
        a_e_kl.append(total)
    
    delta_sigma = [(a - A_E_EXPERIMENT) / A_E_UNCERTAINTY for a in a_e_kl]
    ax.plot(lambda_values, delta_sigma, linewidth=2, label=f'k={k} ({min(k-1,5)} loops)')

ax.axhline(0, color='r', linestyle='--', linewidth=1.5, label='Experiment')
ax.set_xlabel(r'$\lambda$', fontsize=14)
ax.set_ylabel(r'$(a_e - a_e^{\mathrm{exp}}) / \sigma_{\mathrm{exp}}$', fontsize=14)
ax.set_title(r'Deviation from experiment in $\sigma$ units: threshold $k$ × attenuation $\lambda$', fontsize=13)
ax.legend()
ax.set_ylim(-500, 500)
plt.tight_layout()
plt.savefig('../plots/g2_phase_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

print("All lines cross zero near λ ≈ 1 — no non-trivial solution exists.")

## 8. Nesting-Depth Analysis: What Would Be Needed at Higher Loops

At 2-loop we can decompose $C_2$ into skeleton + SE contributions. For a complete test of nesting-depth attenuation, we'd need similar decompositions at 3, 4, 5-loop. What's available in the literature?

In [ ]:
print("Higher-loop nesting decomposition status:")
print("=" * 70)
print()

decomposition_info = {
    1: {
        'total_diagrams': 1,
        'skeleton': 1,
        'se_1': 0,
        'se_2plus': 0,
        'decomposition_known': True,
        'note': 'Schwinger term only — trivially skeleton'
    },
    2: {
        'total_diagrams': 7,
        'skeleton': 4,
        'se_1': 3,
        'se_2plus': 0,
        'decomposition_known': True,
        'note': 'Fully decomposed: II, III(a-c) skeleton; I(a-c) SE-insertion'
    },
    3: {
        'total_diagrams': 72,
        'skeleton': '~50',
        'se_1': '~20',
        'se_2plus': '~2',
        'decomposition_known': False,
        'note': 'Individual diagram results exist (Laporta & Remiddi) but nesting classification needs work'
    },
    4: {
        'total_diagrams': 891,
        'skeleton': '?',
        'se_1': '?',
        'se_2plus': '?',
        'decomposition_known': False,
        'note': 'Numerical results from Aoyama et al. — gauge-invariant subsets identified but nesting depth not published'
    },
    5: {
        'total_diagrams': 12672,
        'skeleton': '?',
        'se_1': '?',
        'se_2plus': '?',
        'decomposition_known': False,
        'note': 'Only total coefficient published'
    },
}

for n, info in decomposition_info.items():
    print(f"{n}-loop: {info['total_diagrams']} diagrams")
    print(f"  Skeleton (σ=0): {info['skeleton']}")
    print(f"  SE nesting=1:   {info['se_1']}")
    print(f"  SE nesting≥2:   {info['se_2plus']}")
    print(f"  Known: {'YES' if info['decomposition_known'] else 'NO'}")
    print(f"  {info['note']}")
    print()

print("CONCLUSION: Full nesting-depth attenuation test requires classifying")
print("the 72 three-loop diagrams by SE-insertion nesting depth and extracting")
print("their individual contributions. This data exists in the literature")
print("(Laporta & Remiddi 1996) but has not been organized for this purpose.")

## 9. Sensitivity Analysis: How Much Can Nesting Attenuation Shift $a_e$?

Even without higher-loop nesting decompositions, we can bound the effect. The 2-loop SE contribution is the only decomposed piece, so let's see what range of $a_e$ values nesting-depth attenuation can produce.

In [ ]:
# The maximum shift from nesting attenuation at 2-loop:
# Going from λ=1 (standard) to λ→∞ (kill all SE) shifts C_2 by -C_2^SE

shift_2loop = -C2_SE * ALPHA_OVER_PI**2  # effect of removing SE at 2-loop

print(f"Maximum shift from 2-loop nesting attenuation:")
print(f"  Δa_e = {shift_2loop:+.6e} = {shift_2loop/A_E_UNCERTAINTY:+.0f}σ")
print()
print(f"Current gap between our series and experiment:")
gap = compute_g2_standard(5) - A_E_EXPERIMENT
print(f"  Δa_e = {gap:+.6e} = {gap/A_E_UNCERTAINTY:+.0f}σ")
print()

# Can nesting attenuation bridge the gap?
# Our series undershoots experiment by ~37σ.
# Removing SE makes it worse (shifts by -415σ). 
# λ < 1 (enhancing SE) could help, but λ < 1 means amplification, not attenuation.
print(f"Direction check:")
print(f"  Our series: {compute_g2_standard(5):.15e}")
print(f"  Experiment: {A_E_EXPERIMENT:.15e}")
print(f"  Series is {'below' if gap < 0 else 'above'} experiment.")
print(f"  SE diagrams contribute positively (+{C2_SE}).")
print(f"  Suppressing SE (λ > 1) moves a_e DOWN — away from experiment.")
print(f"  Enhancing SE (λ < 1) moves a_e UP — toward experiment.")
print(f"  But λ < 1 is amplification, not attenuation — outside the SCN framework.")

## 10. Interpretation and Conclusions

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║              RESULTS: Depth-Weighted Perturbation Theory           ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  APPROACH 1: Loop-order attenuation (σ = loop order n)             ║
║  ─────────────────────────────────────────────────────              ║
║  • Equivalent to rescaling α → α/λ.                               ║
║  • Monotonically decreasing in λ.                                  ║
║  • Optimal λ ≈ 1.000 at all truncation orders.                    ║
║  • VERDICT: Trivial — cannot improve on standard QED.             ║
║                                                                    ║
║  APPROACH 2: Nesting-depth attenuation (σ = SE nesting depth)      ║
║  ─────────────────────────────────────────────────────              ║
║  • Non-trivial: changes internal structure of each loop order.     ║
║  • At 2-loop: suppressing SE diagrams (λ > 1) moves a_e in the    ║
║    WRONG direction (away from experiment).                         ║
║  • Binary SCN (λ→∞) is the 415σ falsification.                    ║
║  • λ < 1 (amplification) could help, but violates SCN semantics.  ║
║  • Higher-loop nesting decompositions needed for full test.        ║
║  • VERDICT: Does not help at 2-loop. Higher loops TBD.            ║
║                                                                    ║
║  STRUCTURAL FINDING                                                ║
║  ─────────────────                                                 ║
║  The QED series converges so well that the 1-loop Schwinger term   ║
║  dominates (99.8% of a_e). Any attenuation scheme that suppresses  ║
║  higher-loop contributions is fighting against an already-optimal  ║
║  convergent series. The attenuation operator N_λ is designed to    ║
║  suppress "deeply self-referential" contributions, but in QED      ║
║  these contributions are both small AND have the right sign.       ║
║                                                                    ║
║  NEXT STEPS                                                        ║
║  ──────────                                                        ║
║  1. Classify 3-loop diagrams by nesting depth (from Laporta &      ║
║     Remiddi 1996 data) and test nesting attenuation at 3-loop.     ║
║  2. Consider QCD observables where the series converges POORLY     ║
║     (e.g., R-ratio, heavy quark production) — attenuation might    ║
║     help regularize asymptotic behavior.                           ║
║  3. Test whether N_λ can serve as a resummation tool for           ║
║     asymptotic/divergent series (complementary to Borel).          ║
║                                                                    ║
╚══════════════════════════════════════════════════════════════════════╝
""")

## Appendix: Raw Numerical Data

In [ ]:
# Dump all key values for future reference
print("Physical constants:")
print(f"  α = {ALPHA:.12e}")
print(f"  α/π = {ALPHA_OVER_PI:.12e}")
print(f"  a_e (exp) = {A_E_EXPERIMENT:.15e} ± {A_E_UNCERTAINTY:.2e}")
print()

print("QED coefficients:")
for n, c in QED_G2_COEFFICIENTS.items():
    print(f"  C_{n} = {c['value']:+.15f} ± {c['uncertainty']:.1e}  [{c['source']}]")
print()

print("2-loop decomposition:")
print(f"  C_2^SE   = {C2_SE:+.12f}  (SE-insertion diagrams I(a,b,c))")
print(f"  C_2^skel = {C2_SKEL:+.12f}  (skeleton diagrams II, III(a-c))")
print()

print("Series sums:")
for n in range(1, 6):
    val = compute_g2_standard(n)
    sig = (val - A_E_EXPERIMENT) / A_E_UNCERTAINTY
    print(f"  Through {n}-loop: {val:.15e}  ({sig:+.1f}σ from exp)")

---

# Part II: QCD Observables

QED converges too well for attenuation to matter. QCD is the natural next target:
the perturbative series for hadronic observables has rapidly growing coefficients
($K_{n+1}/K_n \sim 5$), and the gluon self-energy loop — the most self-referential
contribution to $\beta_0$ — is the dominant piece.

**Observables tested:**
1. $R_\tau$ (hadronic $\tau$ decay) — FOPT coefficients through $\alpha_s^4$
2. $\alpha_s(Q)$ running — 12 experimental points from 1.8 to 206 GeV

**Attenuation strategies:**
- Loop-order: weight $n$-th order term by $\lambda^{-n}$
- Nesting-depth: attenuate the gluon self-energy contribution to $\beta_0$ by $\lambda^{-1}$

In [ ]:
# ── QCD setup ──────────────────────────────────────────────────────
from src.graded_scn import (
    beta0_standard, beta0_attenuated, alpha_s_running_1loop,
    r_tau_coefficients, compute_r_tau_standard, compute_r_tau_attenuated,
    ALPHA_S_MZ, M_Z, M_TAU, R_TAU_EXPERIMENT, V_UD_SQ, S_EW,
    BETA0_GLUON_LOOP, BETA0_GHOST_LOOP, BETA0_QUARK_PER_NF
)
from src.experimental_data import get_alpha_s_data

# β_0 decomposition
print("β_0 decomposition (SU(3), fundamental quarks):")
print(f"  Gluon self-energy loop: {BETA0_GLUON_LOOP}")
print(f"  Ghost loop:             {BETA0_GHOST_LOOP}")
print(f"  Quark (per n_f):        {BETA0_QUARK_PER_NF:.4f}")
print()
for nf in [3, 4, 5]:
    print(f"  β_0(n_f={nf}) = {beta0_standard(nf):.3f}")
print()

# R_τ coefficients
K = r_tau_coefficients()
print("R_τ FOPT coefficients K_n (n_f = 3):")
for n in sorted(K):
    print(f"  K_{n} = {K[n]:>10.3f}")
ratios = [K[n+1]/K[n] for n in sorted(K) if n+1 in K]
print(f"  Growth ratios K_{{n+1}}/K_n: {[f'{r:.2f}' for r in ratios]}")
print(f"  → Series grows ~5× per loop order (poorly convergent)")

## $R_\tau$: Standard vs Attenuated

$R_\tau = 3|V_{ud}|^2 S_\text{EW}(1 + \delta_\text{pert} + \delta_\text{NP})$ where the perturbative correction is $\delta_\text{pert} = \sum_{n=1}^{N} K_n (\alpha_s/\pi)^n$.

We test two attenuation strategies:
1. **Loop-order**: weight $K_n \to K_n / \lambda^n$ (direct series suppression)
2. **Nesting-depth $\beta_0$**: attenuate gluon self-energy in $\beta_0$, changing $\alpha_s(m_\tau)$

In [ ]:
# ── R_τ: standard calculation ──────────────────────────────────────
as_mtau = alpha_s_running_1loop(M_TAU**2, 3, beta0_standard(3))
print(f"αs(m_τ) from 1-loop running: {as_mtau:.4f}")
print(f"β_0(n_f=3) = {beta0_standard(3):.3f}")
print()

r_std = compute_r_tau_standard(as_mtau)
print(f"R_τ standard (FOPT through αs⁴): {r_std['r_tau']:.4f}")
print(f"R_τ experimental:                {R_TAU_EXPERIMENT}")
print(f"Deviation: {(r_std['r_tau'] - R_TAU_EXPERIMENT)/0.011:.1f}σ")
print()

# Individual term contributions
a_pi = as_mtau / np.pi
print("Perturbative series terms:")
for n in sorted(r_std['terms']):
    print(f"  n={n}: K_{n} × (αs/π)^{n} = {K[n]:.3f} × {a_pi**n:.6f} = {r_std['terms'][n]:.6f}")
print(f"  δ_pert = {r_std['delta_pert']:.6f}")
print()
print("Note: αs(m_τ)=0.352 from 1-loop overshoots world avg ~0.330.")
print("This is because 1-loop running doesn't capture higher-order effects.")
print("The R_τ value computed here is inflated relative to higher-order analyses.")

In [ ]:
# ── Loop-order attenuation stability test ──────────────────────────
# Key test: does a single λ work across truncation orders?

print("Loop-order attenuation on R_τ:")
print(f"{'max_order':>9}  {'λ*':>6}  {'R_τ(std)':>10}  {'R_τ(att)':>10}")
print("-" * 45)

optimal_lambdas = []
for max_ord in [1, 2, 3, 4]:
    best_lam = 1.0
    best_diff = float('inf')
    for lam in np.linspace(0.5, 3.0, 500):
        r = compute_r_tau_attenuated(as_mtau, lam, max_ord)['r_tau']
        diff = abs(r - R_TAU_EXPERIMENT)
        if diff < best_diff:
            best_diff = diff
            best_lam = lam
    r_std_ord = compute_r_tau_standard(as_mtau, max_ord)['r_tau']
    r_att_ord = compute_r_tau_attenuated(as_mtau, best_lam, max_ord)['r_tau']
    optimal_lambdas.append(best_lam)
    print(f"  O(αs^{max_ord})    {best_lam:.3f}  {r_std_ord:>10.4f}  {r_att_ord:>10.4f}")

print()
print(f"λ* sequence: {[f'{l:.3f}' for l in optimal_lambdas]}")
print(f"λ* range: {min(optimal_lambdas):.3f} → {max(optimal_lambdas):.3f}")
print()
print("VERDICT: λ* is NOT stable across truncation orders.")
print("It shifts from ~0.6 at O(αs) to ~1.1 at O(αs⁴).")
print("This is just curve-fitting: no single λ produces a")
print("consistent depth-weighted theory.")

## $\alpha_s(Q)$ running: standard 1-loop vs attenuated $\beta_0$

The gluon self-energy loop contributes $10$ to $\beta_0 = 11 - 2n_f/3$. Under nesting-depth
attenuation (the gluon correcting itself is the prototypical $\sigma = 1$ diagram), we get:

$$\beta_0(\lambda) = \frac{10}{\lambda} + 1 - \frac{2n_f}{3}$$

The 1-loop exact running formula (with our convention $\beta_0 = 11 - 2n_f/3$):

$$\alpha_s(Q^2) = \frac{\alpha_s(M_Z^2)}{1 + \frac{\beta_0 \alpha_s(M_Z^2)}{4\pi} \ln\frac{Q^2}{M_Z^2}}$$

In [ ]:
# ── αs running: comparison with experimental data ─────────────────
as_data = get_alpha_s_data()

# Standard 1-loop
print(f"{'Q (GeV)':>8}  {'1L-std':>8}  {'exp':>8}  {'err':>8}  {'pull':>6}")
print("-" * 44)
chi2_std = 0.0
n_std = 0
for Q, as_exp, as_err in zip(as_data['Q'], as_data['alpha_s'], as_data['alpha_s_err']):
    nf = 3 if Q < 1.5 else (4 if Q < 4.5 else (5 if Q < 175 else 6))
    a_std = alpha_s_running_1loop(Q**2, nf, beta0_standard(nf))
    if not np.isnan(a_std):
        pull = (a_std - as_exp) / as_err
        chi2_std += pull**2
        n_std += 1
        print(f"{Q:>8.1f}  {a_std:>8.4f}  {as_exp:>8.4f}  {as_err:>8.4f}  {pull:>+6.2f}")
    else:
        print(f"{Q:>8.1f}       nan  {as_exp:>8.4f}  {as_err:>8.4f}")

ndf = n_std - 1
print(f"\nχ²/ndf = {chi2_std/ndf:.2f}  ({n_std} points, ndf={ndf})")
print("\nStandard 1-loop running already describes the data well (χ²/ndf < 1).")

In [ ]:
# ── Attenuated β_0: scan for optimal λ ────────────────────────────
print("Scanning λ for attenuated 1-loop αs(Q) running:")
print()

best_chi2 = float('inf')
best_lam = 1.0
lam_range = np.linspace(0.5, 5.0, 500)
chi2_vs_lam = []

for lam in lam_range:
    chi2 = 0.0
    n = 0
    for Q, as_exp, as_err in zip(as_data['Q'], as_data['alpha_s'], as_data['alpha_s_err']):
        nf = 3 if Q < 1.5 else (4 if Q < 4.5 else (5 if Q < 175 else 6))
        a = alpha_s_running_1loop(Q**2, nf, beta0_attenuated(nf, lam))
        if not np.isnan(a):
            chi2 += ((a - as_exp) / as_err)**2
            n += 1
    if n > 1:
        chi2_ndf = chi2 / (n - 1)
        chi2_vs_lam.append((lam, chi2_ndf))
        if chi2_ndf < best_chi2:
            best_chi2 = chi2_ndf
            best_lam = lam

nf5 = 5  # dominant range
print(f"Optimal λ* = {best_lam:.3f}")
print(f"β_0(n_f={nf5}, λ*) = {beta0_attenuated(nf5, best_lam):.3f}  vs  standard {beta0_standard(nf5):.3f}")
print(f"χ²/ndf at λ*:    {best_chi2:.2f}")
print(f"χ²/ndf at λ=1:   {chi2_std/ndf:.2f}")
print()

if best_lam < 1.0:
    print(f"⚠ λ* = {best_lam:.3f} < 1: the fit prefers ENHANCED gluon loops,")
    print("  the OPPOSITE of what SCN (suppression of self-reference) predicts.")
elif abs(best_lam - 1.0) < 0.1:
    print(f"λ* ≈ 1: attenuation has negligible effect. Standard theory already optimal.")
else:
    print(f"λ* = {best_lam:.3f}: modest suppression of gluon self-energy.")

# Plot χ² vs λ
fig, ax = plt.subplots(figsize=(8, 4))
lams, chi2s = zip(*chi2_vs_lam)
ax.plot(lams, chi2s, 'b-')
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.5, label='λ=1 (standard)')
ax.axvline(best_lam, color='red', linestyle='--', alpha=0.5, label=f'λ*={best_lam:.3f}')
ax.axhline(1.0, color='green', linestyle=':', alpha=0.3, label='χ²/ndf=1')
ax.set_xlabel('λ')
ax.set_ylabel('χ²/ndf')
ax.set_title('αs running: χ²/ndf vs attenuation parameter λ')
ax.legend()
ax.set_xlim(0.5, 3.0)
ax.set_ylim(0, 3)
plt.tight_layout()
plt.show()

## QCD Conclusions

### What we tested

| Test | Observable | Strategy | Result |
|------|-----------|----------|--------|
| 1 | $R_\tau$ | Loop-order $\mathcal{N}_\lambda$ | λ* unstable: 0.57 → 1.10 across orders |
| 2 | $R_\tau$ | Nesting $\beta_0$ | λ* ≈ 1.05 (negligible effect) |
| 3 | $\alpha_s(Q)$ | Nesting $\beta_0$ | λ* < 1 (wrong direction) or ≈ 1 |

### Diagnosis

1. **Loop-order attenuation is just curve-fitting.** A consistent physical parameter $\lambda$ should be stable across truncation orders. It isn't: the optimal $\lambda$ for $R_\tau$ varies by a factor of 2 depending on how many orders we include.

2. **Nesting-depth $\beta_0$ attenuation is ineffective.** Standard 1-loop running already fits $\alpha_s(Q)$ data with $\chi^2/\text{ndf} \approx 0.77$. The optimal attenuation parameter is either $\lambda < 1$ (enhanced gluon loops — contradicts SCN's premise) or $\lambda \approx 1$ (no attenuation needed).

3. **The real issue isn't the framework — it's that perturbation theory already works.** Both QED and QCD are well-described by standard perturbative methods. The graded SCN attenuation operator $\mathcal{N}_\lambda$ is designed to help with poorly-behaved series, but these series aren't poorly enough behaved to need external help.

### Where attenuation might still matter

Genuinely asymptotic/divergent series where Borel summation is needed — e.g., the large-order behavior of perturbation theory ($K_n \sim n!$), or observables sensitive to the renormalon ambiguity. At the currently computed orders ($n \leq 4$), the QCD series has not yet entered the asymptotic regime.